In [0]:
from pyspark.sql.functions import col
df_silver = spark.read.table("he_prod_incen_analytics_dbw_01.silver.fact_claims")

# Read Gold Dimensions
df_dim_members = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_members")
df_dim_providers = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_providers")
df_dim_facilities = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_facilities")
df_dim_date = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_date")
df_dim_service = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_service_types")
df_dim_icd10 = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_icd10_codes")
df_dim_cpt = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_cpt_codes")
df_dim_plan = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_plan_types")
df_dim_network = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_network_status")
df_dim_claim_status = spark.read.table("he_prod_incen_analytics_dbw_01.gold.dim_claim_status")

In [0]:
from pyspark.sql.functions import monotonically_increasing_id, coalesce, lit, col

# 1. Generate Fact Surrogate Key
df_gold = df_silver.withColumn("claim_sk", monotonically_increasing_id())

# 2. DIMENSION LOOKUPS & ORPHAN HANDLING (Using Business Keys)
df_gold = df_gold.join(df_dim_members.select(col("member_sk").alias("gold_member_sk"), "member_id"), on="member_id", how="left") \
                 .withColumn("gold_member_sk", coalesce(col("gold_member_sk"), lit(-1)))

df_gold = df_gold.join(df_dim_providers.select(col("provider_sk").alias("gold_provider_sk"), "provider_id"), on="provider_id", how="left") \
                 .withColumn("gold_provider_sk", coalesce(col("gold_provider_sk"), lit(-1)))

df_gold = df_gold.join(df_dim_facilities.select(col("facility_sk").alias("gold_facility_sk"), "facility_id"), on="facility_id", how="left") \
                 .withColumn("gold_facility_sk", coalesce(col("gold_facility_sk"), lit(-1)))

# Date lookup on service_date
df_gold = df_gold.join(df_dim_date.select(col("date_sk").alias("gold_date_sk"), col("date").alias("service_date")), on="service_date", how="left") \
                 .withColumn("gold_date_sk", coalesce(col("gold_date_sk"), lit(-1)))

df_gold = df_gold.join(df_dim_service.select(col("service_type_sk").alias("gold_service_type_sk"), "service_type_id"), on="service_type_id", how="left") \
                 .withColumn("gold_service_type_sk", coalesce(col("gold_service_type_sk"), lit(-1)))

df_gold = df_gold.join(df_dim_icd10.select(col("icd10_sk").alias("gold_icd10_sk"), "icd10_code"), on="icd10_code", how="left") \
                 .withColumn("gold_icd10_sk", coalesce(col("gold_icd10_sk"), lit(-1)))

df_gold = df_gold.join(df_dim_cpt.select(col("cpt_sk").alias("gold_cpt_sk"), "cpt_code"), on="cpt_code", how="left") \
                 .withColumn("gold_cpt_sk", coalesce(col("gold_cpt_sk"), lit(-1)))

df_gold = df_gold.join(df_dim_plan.select(col("plan_sk").alias("gold_plan_sk"), "plan_type_id"), on="plan_type_id", how="left") \
                 .withColumn("gold_plan_sk", coalesce(col("gold_plan_sk"), lit(-1)))

df_gold = df_gold.join(df_dim_network.select(col("network_status_sk").alias("gold_network_status_sk"), "network_status_id"), on="network_status_id", how="left") \
                 .withColumn("gold_network_status_sk", coalesce(col("gold_network_status_sk"), lit(-1)))

df_gold = df_gold.join(df_dim_claim_status.select(col("claim_status_sk").alias("gold_claim_status_sk"), "claim_status_id"), on="claim_status_id", how="left") \
                 .withColumn("gold_claim_status_sk", coalesce(col("gold_claim_status_sk"), lit(-1)))

# 3. Column Pruning - Select ONLY Gold required columns
df_gold = df_gold.select(
    "claim_sk", "claim_id",
    col("gold_member_sk").alias("member_sk"),
    col("gold_provider_sk").alias("provider_sk"),
    col("gold_facility_sk").alias("facility_sk"),
    col("gold_date_sk").alias("service_date_sk"),
    col("gold_service_type_sk").alias("service_type_sk"),
    col("gold_icd10_sk").alias("icd10_code_sk"),
    col("gold_cpt_sk").alias("cpt_code_sk"),
    col("gold_plan_sk").alias("plan_type_sk"),
    col("gold_network_status_sk").alias("network_status_sk"),
    col("gold_claim_status_sk").alias("claim_status_sk"),
    "total_charge", "contractual_adjustment", "allowed_amount", 
    "deductible_applied", "co_pay_amount", "copay_applied", 
    "insurance_paid_amount", "member_responsibility", "provider_write_off",
    "denial_reason", "prior_auth_required", "prior_auth_obtained", 
    "emergency_service", "preventive_service", "special_handling",
    "claim_type", "days_to_process", "days_to_pay", 
    "units_of_service", "quantity", "appeal_flag", "appeal_amount", "appeal_status"
)

display(df_gold.limit(5))

In [0]:
gold_table_fqn = "he_prod_incen_analytics_dbw_01.gold.fact_claims"

df_gold.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(gold_table_fqn)

print(f"✅ Successfully wrote Fact_Claims to Gold layer.")